# IP&MC FIRM reranking extension

Runs BGE + hybrid RRF with and without cross-encoder reranking on the same questions, seeds, and chunkers. Select a GPU runtime, then run all cells.

In [ ]:
import subprocess

gpu = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True,
    text=True,
    check=False,
)
if gpu.returncode != 0:
    raise RuntimeError("No GPU detected. In Colab, choose Runtime > Change runtime type > GPU.")
print(gpu.stdout.strip())

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import os
import subprocess
import sys

REPO = "/content/chunkrag-course-project"
REPOSITORY_URL = "https://github.com/kuromi1kow/chunkrag-course-project.git"

if not os.path.isdir(os.path.join(REPO, ".git")):
    subprocess.run(["git", "clone", REPOSITORY_URL, REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "fetch", "origin", "main"], check=True)
    subprocess.run(["git", "-C", REPO, "checkout", "main"], check=True)
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only", "origin", "main"], check=True)

commit = subprocess.check_output(["git", "-C", REPO, "rev-parse", "HEAD"], text=True).strip()
print(f"Running commit: {commit}")

packages = [
    "datasets==2.21.0",
    "faiss-cpu==1.14.3",
    "huggingface-hub==0.36.2",
    "langchain-text-splitters==0.3.11",
    "numpy==1.26.4",
    "rank-bm25==0.2.2",
    "sentence-transformers==3.4.1",
    "sentencepiece==0.2.1",
    "spacy==3.8.14",
    "transformers==4.57.6",
    "tqdm==4.68.4",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--no-deps", "--ignore-requires-python", "-e", REPO],
    check=True,
)
sys.path.insert(0, os.path.join(REPO, "src"))

In [ ]:
from pathlib import Path

CONFIG = Path(REPO) / "configs" / "ipmc_firm_rerank_bge.json"
OUTPUT = Path("/content/drive/MyDrive/chunkrag_outputs/ipmc_firm_rerank_bge")
OVERWRITE = False

if OUTPUT.exists() and any(OUTPUT.iterdir()) and not OVERWRITE:
    raise RuntimeError(
        f"Output already exists at {OUTPUT}. Set OVERWRITE=True only if the existing run should be replaced."
    )

command = [
    sys.executable,
    str(Path(REPO) / "scripts" / "run_experiments.py"),
    "--config",
    str(CONFIG),
    "--output-dir",
    str(OUTPUT),
]
if OVERWRITE:
    command.append("--overwrite")

subprocess.run(command, cwd=REPO, check=True)
print(f"Completed reranking artifacts: {OUTPUT}")

In [ ]:
import json

manifest = json.loads((OUTPUT / "run_manifest.json").read_text())
summaries = json.loads((OUTPUT / "all_results.json").read_text())
assert manifest["status"] == "complete", manifest
assert len(summaries) == 72, len(summaries)
assert {row["retriever"] for row in summaries} == {"hybrid", "hybrid_rerank"}
assert {row["seed"] for row in summaries} == {13, 21, 34}
print("Validation passed: 72 paired hybrid and reranked multi-seed cells.")